# Strategy 1 — Structured Prompt Reasoner
**Topic:** LLM-based reasoning strategies in FrozenLake-v1 · **Platform:** Google Colab — **GPU T4**

**Focus: prompt design.** Only the phrasing of the prompt changes.

| | |
|---|---|
| Memory | none |
| Decoding | greedy, `temperature = 0` |
| Compared | 4 prompt phrasings |

## 1. Setup
Model: **`Qwen/Qwen2.5-1.5B-Instruct`** (open weights, fits a T4 GPU).

In [ ]:
%pip install -q -U gymnasium transformers accelerate

In [ ]:
import os
import re
import textwrap
import time
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import logging as hf_logging
from transformers import set_seed as hf_set_seed

hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)

SEED = 42
hf_set_seed(SEED)                         # seeds random, numpy and torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print("PyTorch", torch.__version__, "| gymnasium", gym.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Model
- `temperature = 0` → greedy decoding (`do_sample=False`, deterministic).
- `temperature > 0` → sampling with `top_p`.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")   # left padding for batched generation
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda").eval()
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters")

In [ ]:
@torch.no_grad()
def generate(conversations, max_new_tokens=256, temperature=0.0, top_p=1.0, seed=SEED, batch_size=32):
    # Batch of chat conversations → one output (text, number of tokens) per conversation.
    hf_set_seed(seed)
    prompts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations]
    if temperature > 0:
        sampling = dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=0)
    else:
        sampling = dict(do_sample=False)
    results = []
    for b in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[b:b + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=max_new_tokens, repetition_penalty=1.0,
                             pad_token_id=tokenizer.pad_token_id, **sampling)
        for g in out[:, enc.input_ids.shape[1]:]:
            ids = g[g != tokenizer.pad_token_id].tolist()
            results.append({"text": tokenizer.decode(ids, skip_special_tokens=True).strip(), "n_tokens": len(ids)})
    return results

print(generate([[{"role": "user", "content": "In one sentence: what is the FrozenLake environment?"}]], max_new_tokens=60)[0]["text"])

## 3. FrozenLake environment
`is_slippery=False`. Each run uses **10 maps** (the default 4x4 map + 9 random maps with fixed seeds), so the results do not depend on a single map. In every map S is the top-left corner and G the bottom-right corner. The LLM sees the grid with the agent marked **A**.

In [ ]:
N_MAPS = 10
MAX_STEPS = 15     # step limit per episode

DEFAULT_MAP = ["SFFF", "FHFH", "FFFH", "HFFG"]
MAPS, seed = [{"id": "default", "desc": DEFAULT_MAP}], 0
while len(MAPS) < N_MAPS:                     # random maps without duplicates
    seed += 1
    desc = generate_random_map(size=4, p=0.7, seed=seed)
    if desc not in [m["desc"] for m in MAPS]:
        MAPS.append({"id": f"rand{seed}", "desc": desc})
MAP_BY_ID = {m["id"]: m for m in MAPS}
GOAL = (3, 3)

ACTIONS = ["LEFT", "DOWN", "RIGHT", "UP"]           # Gymnasium order: 0, 1, 2, 3
MOVES = [(0, -1), (1, 0), (0, 1), (-1, 0)]
TILE_NAMES = {"S": "the starting square (solid ice)", "F": "solid ice", "H": "a HOLE, you would fall in", "G": "the GOAL"}

def target(desc, pos, a):
    # Cell reached by action a (moving into the border leaves the agent in place).
    r, c = pos[0] + MOVES[a][0], pos[1] + MOVES[a][1]
    return (r, c) if 0 <= r < len(desc) and 0 <= c < len(desc) else pos

def goal_distances(desc):
    # BFS from G (only for evaluation): minimum steps from every cell to the goal.
    dist, queue = {GOAL: 0}, deque([GOAL])
    while queue:
        cur = queue.popleft()
        for a in range(4):
            nxt = target(desc, cur, a)
            if nxt not in dist and desc[nxt[0]][nxt[1]] != "H":
                dist[nxt] = dist[cur] + 1
                queue.append(nxt)
    return dist

def state_text(desc, pos=None):
    # The lake as text. With pos, the agent is marked A and its position is described.
    rows = [list(row) for row in desc]
    if pos is not None:
        rows[pos[0]][pos[1]] = "A"
    grid = "\n".join(["    c0 c1 c2 c3"] + [f"r{r}  " + "  ".join(row) for r, row in enumerate(rows)])
    if pos is None:
        return grid
    return (f"Here is the lake right now (A is you, S is the start, F is solid ice, H is a hole, G is the goal):\n"
            f"{grid}\nYou're on row {pos[0]}, col {pos[1]}, and the goal is on row {GOAL[0]}, col {GOAL[1]}.")

def neighbors_text(desc, pos):
    lines = []
    for a, name in enumerate(ACTIONS):
        nxt = target(desc, pos, a)
        if nxt == pos:
            lines.append(f"- {name}: that's the edge of the lake, you'd stay where you are")
        else:
            lines.append(f"- {name}: you'd land on row {nxt[0]}, col {nxt[1]}, which is {TILE_NAMES[desc[nxt[0]][nxt[1]]]}")
    return "\n".join(lines)

for m in MAPS:
    print(f"{m['id']:8s}", " | ".join(m["desc"]))
print("\n" + state_text(DEFAULT_MAP, (1, 0)))

## 4. Action parser
The model must end with `ACTION: <LEFT|DOWN|RIGHT|UP>`. If no valid action is found, the step is a **parse failure**: the agent stays in place (no random fallback action, so every success comes from the model's own decision).

In [ ]:
ACTION_TAG = re.compile(r"\bACTION\s*\**\s*[:=]\s*[\*\[<\s]*(LEFT|DOWN|RIGHT|UP)\b", re.IGNORECASE)
ACTION_WORD = re.compile(r"\b(LEFT|DOWN|RIGHT|UP)\b", re.IGNORECASE)

def parse_action(text):
    tags = ACTION_TAG.findall(text)                      # last 'ACTION:' tag
    if tags:
        return ACTIONS.index(tags[-1].upper())
    words = {w.upper() for w in ACTION_WORD.findall(text)}
    return ACTIONS.index(words.pop()) if len(words) == 1 else None   # or a single action word

for t in ["ACTION: RIGHT", "UP hits a hole, so I go RIGHT.\nACTION: <RIGHT>", "RIGHT", "UP would hit a hole, DOWN is safe"]:
    a = parse_action(t)
    print(f"{t!r:55} → {ACTIONS[a] if a is not None else None}")

## 5. Agent loop and metrics
```
state → prompt → LLM → text → parser → action → env.step → new state  (until G, H or MAX_STEPS)
```
All episodes of a run advance together (one LLM batch per step). Figures are saved in `figures/` (PDF + PNG).

| Metric | Meaning |
|---|---|
| success rate | episodes that reach G |
| hole / timeout rate | episodes that fall into H / reach `MAX_STEPS` |
| steps (success) | mean steps of the successful episodes |
| optimal-move rate | steps that get closer to G along the shortest path |
| parse-fail rate | steps without a valid action |
| wall-bump rate | actions into the border |
| tokens/step | generated tokens per decision (cost of the reasoning) |
| consistency | repeats of a map that follow the same path (reproducibility) |

In [ ]:
def llm_decide(episodes, build_messages, gen_cfg, seed):
    outs = generate([build_messages(e) for e in episodes], seed=seed, **gen_cfg)
    return [{"text": o["text"], "n_tokens": o["n_tokens"], "action": parse_action(o["text"])} for o in outs]

def run_episodes(maps, build_messages, gen_cfg, n_repeats=1, decide=llm_decide, label=""):
    episodes = []
    for m in maps:
        for rep in range(n_repeats):
            env = gym.make("FrozenLake-v1", desc=m["desc"], is_slippery=False)
            env.reset(seed=SEED)
            episodes.append({"map": m["id"], "desc": m["desc"], "rep": rep, "env": env, "pos": (0, 0),
                             "dist": goal_distances(m["desc"]), "trace": [], "done": False, "outcome": "timeout"})
    t0 = time.time()
    for step in range(MAX_STEPS):
        active = [e for e in episodes if not e["done"]]
        if not active:
            break
        for e, d in zip(active, decide(active, build_messages, gen_cfg, SEED + step)):
            pos, a = e["pos"], d["action"]
            if a is None:                              # parse failure: stay in place
                nxt, terminated = pos, False
            else:
                obs, _, terminated, _, _ = e["env"].step(a)
                nxt = divmod(int(obs), len(e["desc"]))
            tile = e["desc"][nxt[0]][nxt[1]]
            e["trace"].append({"step": step, "pos": pos, "action": None if a is None else ACTIONS[a], "next": nxt,
                               "tile": tile, "text": d["text"], "n_tokens": d["n_tokens"], "parse_fail": a is None,
                               "wall": a is not None and nxt == pos,
                               "optimal": e["dist"].get(nxt, -1) == e["dist"][pos] - 1, **d.get("extra", {})})
            e["pos"] = nxt
            if terminated:
                e["done"], e["outcome"] = True, ("goal" if tile == "G" else "hole")
    eps = pd.DataFrame([{"run": label, "map": e["map"], "rep": e["rep"], "outcome": e["outcome"],
                         "success": e["outcome"] == "goal", "steps": len(e["trace"]),
                         "path": "".join(t["action"][0] if t["action"] else "?" for t in e["trace"])} for e in episodes])
    steps = pd.DataFrame([{"run": label, "map": e["map"], "rep": e["rep"], **t} for e in episodes for t in e["trace"]])
    eps.attrs["seconds"] = time.time() - t0
    return eps, steps

In [ ]:
def summary_table(runs):
    rows = {}
    for name, (eps, steps) in runs.items():
        succ = eps[eps["success"]]
        rows[name] = {
            "success rate": eps["success"].mean(),
            "hole rate": (eps["outcome"] == "hole").mean(),
            "timeout rate": (eps["outcome"] == "timeout").mean(),
            "steps (success)": succ["steps"].mean() if len(succ) else np.nan,
            "optimal-move rate": steps["optimal"].mean(),
            "parse-fail rate": steps["parse_fail"].mean(),
            "wall-bump rate": steps["wall"].mean(),
            "tokens/step": steps["n_tokens"].mean(),
            "consistency": eps.groupby("map")["path"].agg(lambda p: p.value_counts().iloc[0] / len(p)).mean(),
            "seconds": eps.attrs.get("seconds", np.nan),
        }
    return pd.DataFrame(rows).T.astype(float).round(3)

os.makedirs("figures", exist_ok=True)

def plot_results(summary, title="", name=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.4 * len(summary) + 2.2))
    summary[["success rate", "hole rate", "timeout rate"]].plot.barh(
        stacked=True, ax=axes[0], color=["tab:green", "tab:red", "tab:gray"])
    summary[["optimal-move rate", "parse-fail rate", "wall-bump rate"]].plot.barh(
        ax=axes[1], color=["tab:blue", "tab:orange", "tab:purple"])
    for ax, subtitle in zip(axes, ["Episode outcome", "Step-level decisions"]):
        ax.set_xlim(0, 1); ax.invert_yaxis(); ax.grid(alpha=0.3, axis="x"); ax.set_title(subtitle); ax.legend(fontsize=8)
    axes[1].set_yticklabels([])
    fig.suptitle(title); plt.tight_layout()
    if name:                                   # PDF (vector, for LaTeX) + PNG (300 dpi)
        fig.savefig(f"figures/{name}.pdf", bbox_inches="tight")
        fig.savefig(f"figures/{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

def show_transcript(steps, map_id, rep=0, max_chars=700):
    s = steps[(steps["map"] == map_id) & (steps["rep"] == rep)]
    print(state_text(MAP_BY_ID[map_id]["desc"], s["pos"].iloc[0]), "\n")
    for _, t in s.iterrows():
        print(f"━━━ step {t['step']} | {t['pos']} → {t['action']} → {t['next']} ({t['tile']}) | {t['n_tokens']} tokens ━━━")
        print(textwrap.indent(t["text"][:max_chars] + (" [...]" if len(t["text"]) > max_chars else ""), "  "), "\n")

## 6. Prompts
| Prompt | Idea | Reasoning |
|---|---|---|
| `direct` | answer only with the action | implicit |
| `step_by_step` | "Think step by step and decide the safest action" | explicit |
| `rules` | + movement rules, check each action | explicit, guided |
| `structured` | + what each action leads to, fixed answer format | explicit, structured |

In [ ]:
TASK = "You need to reach the goal (G) without falling into any hole (H). You can move LEFT, RIGHT, UP or DOWN."
FORMAT = "Once you've made up your mind, finish with one last line that says ACTION: followed by your move."
RULES = """A few things to keep in mind:
- UP takes you one row up (row - 1) and DOWN one row down (row + 1); LEFT and RIGHT move you one column (col - 1 / col + 1).
- If you try to walk off the edge of the lake, you just stay where you are.
- Stepping on a hole means you fall in and it's game over. Reaching G means you made it.
- Whenever you can, pick a move that brings you closer to G."""

T_DIRECT = """You're crossing a frozen lake.
{state}
{task}
No need to explain anything, just tell me your next move. Reply only with ACTION: followed by the move."""

T_STEP_BY_STEP = """You're crossing a frozen lake.
{state}
{task}
Think it through step by step and pick the safest move.
{format}"""

T_RULES = """You're crossing a frozen lake, so you have to be careful where you step.
{state}
{task}
{rules}
Before deciding, go through each move: which square would it take you to, and is that square safe? Then go with the safest move that gets you closer to G.
{format}"""

T_STRUCTURED = """You're crossing a frozen lake, so you have to be careful where you step.
{state}
{task}
{rules}
To make it easier, this is where each move would take you from where you're standing:
{neighbors}
Please answer in this short format:
Safe moves: <moves that don't drop you in a hole or push you against the edge>
Closer to G: <which of those safe moves get you nearer the goal>
My choice: <one short sentence on why>
ACTION: <your move>"""

def prompt_builder(template):
    def build(e):
        return [{"role": "user", "content": template.format(
            state=state_text(e["desc"], e["pos"]), neighbors=neighbors_text(e["desc"], e["pos"]),
            task=TASK, rules=RULES, format=FORMAT)}]
    return build

PROMPTS = {
    "direct":       {"build": prompt_builder(T_DIRECT),       "max_new_tokens": 10},
    "step_by_step": {"build": prompt_builder(T_STEP_BY_STEP), "max_new_tokens": 256},
    "rules":        {"build": prompt_builder(T_RULES),        "max_new_tokens": 256},
    "structured":   {"build": prompt_builder(T_STRUCTURED),   "max_new_tokens": 256},
}

print(PROMPTS["structured"]["build"]({"desc": DEFAULT_MAP, "pos": (0, 0)})[0]["content"])

## 7. Experiment
Each prompt on the same 10 maps, `temperature = 0`.

In [ ]:
runs = {}
for name, p in PROMPTS.items():
    runs[name] = run_episodes(MAPS, p["build"], dict(temperature=0.0, max_new_tokens=p["max_new_tokens"]), label=name)
    print(f"done: {name:13s} ({runs[name][0].attrs['seconds']:.0f} s)")

## 8. Results

In [ ]:
summary = summary_table(runs)
display(summary)
plot_results(summary, "Strategy 1 — prompt phrasing (temperature = 0)", name="s1_prompts")

## 9. Reasoning transcripts

In [ ]:
SHOW_RUN, SHOW_MAP = "rules", "default"
show_transcript(runs[SHOW_RUN][1], SHOW_MAP)

## 10. Save results
`summary_s1.csv` is used in Strategy 3 to compare the three strategies.

In [ ]:
summary.assign(strategy="S1").to_csv("summary_s1.csv")
pd.concat([steps for _, steps in runs.values()]).to_csv("steps_s1.csv", index=False)
print("saved: summary_s1.csv, steps_s1.csv")